# 13 — Model 4: Full Fine-Tuned RoBERTa

Full end-to-end fine-tune with HF Trainer + EarlyStopping.

**Runtime:** T4 GPU (16 GB VRAM). Config uses `per_device_train_batch_size=8` + `gradient_accumulation_steps=2` (effective batch 16) to fit within 16 GB with fp16. Expect ~2–3 hours for 5 epochs over the full dataset.

**Tip:** Run the pilot cell first (10% data, 1 epoch) to confirm Trainer integration before the full run.

In [ ]:
import os, sys
os.chdir('/content/polygence'); sys.path.insert(0, 'src')
from prompt_classifier.seeds import set_all_seeds; set_all_seeds(42)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    assert 'T4' in torch.cuda.get_device_name(0) or True, 'Expected T4 runtime'

In [ ]:
# ---- PILOT: smoke-test on 10% of data, 1 epoch ----
# Verifies Trainer integration before the 2-3h full run.
import pandas as pd
from prompt_classifier.config import load_config
from prompt_classifier.models.m4_roberta_ft import _PromptDataset, _compute_metrics
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

cfg = load_config('configs/model4_roberta_ft.yaml')
train_df = pd.read_parquet(cfg['data']['train_path']).sample(frac=0.10, random_state=42)
val_df   = pd.read_parquet(cfg['data']['val_path']).sample(frac=0.10, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
model = AutoModelForSequenceClassification.from_pretrained(cfg['base_model'], num_labels=2)
args = TrainingArguments(
    output_dir='/tmp/m4_pilot', num_train_epochs=1,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    evaluation_strategy='epoch', save_strategy='no',
    logging_steps=20, fp16=True, report_to='none',
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=_PromptDataset(train_df['prompt'].tolist(), train_df['y'].tolist(), tokenizer),
    eval_dataset=_PromptDataset(val_df['prompt'].tolist(), val_df['y'].tolist(), tokenizer),
    compute_metrics=_compute_metrics,
)
trainer.train()
print('Pilot complete — VRAM looks OK, proceed to full run.')

In [ ]:
# ---- FULL RUN (~2-3h on T4) ----
from prompt_classifier.models.m4_roberta_ft import run
report = run('configs/model4_roberta_ft.yaml')
print(report)

In [ ]:
print('FPR target met:', report['targets_met']['fpr_ok'])
print('FNR target met:', report['targets_met']['fnr_ok'])